In [1]:
import os
import json
from datasets import load_dataset

import sys
sys.set_int_max_str_digits(999999)

In [2]:
taco_dataset = load_dataset('/home/kaixin/Desktop/mmcode/TACO')

/home/kaixin/anaconda3/envs/mmcode/lib/python3.11/site-packages/datasets/load.py:922: FutureWarning: The repository for TACO contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at /home/kaixin/Desktop/mmcode/TACO/TACO.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


In [ ]:
def find_problem_in_taco(problem_id, problem_index):
    for problem in taco_dataset["train"]:
        if not problem.get("url"):
            continue
        if f"{problem_id}/{problem_index}" in problem["url"]:
            return problem
        
    for problem in taco_dataset["train"]:
        if not problem.get("url"):
            continue
        if f"{problem_id}/{problem_index}" in problem["url"]:
            return problem
    return None

In [ ]:
results = []
for path in cf_problems:
    mmcode_data = load_crawled_problems(path)
    problem_id, problem_index = path.split("_")[-2:]
    taco_data = find_problem_in_taco(problem_id, problem_index)
    results.append((path, taco_data))

In [3]:
taco_dataset_dict = {}

for index, item in enumerate(taco_dataset["train"]):
    identifier = f"train_{index}"
    spec = item["question"]
    taco_dataset_dict[identifier] = item

for index, item in enumerate(taco_dataset["test"]):
    identifier = f"test_{index}"
    spec = item["question"]
    taco_dataset_dict[identifier] = item

In [ ]:
for path, taco_data in results:
    filename = os.path.join(path, "data.json")
    with open(filename, 'r') as file:
        data = json.load(file)

    if taco_data is None:
        taco_data = {
            "solutions": None,
            "input_output": None,
            "difficulty": None,
            "tags": None,
            "skill_types": None,
            "raw_tags": None,
            "date": None,
            "Expected Auxiliary Space": None,
            "Expected Time Complexity": None
        }
    if taco_data["solutions"]:
        data["solutions"] = taco_data["solutions"]
    if taco_data["input_output"]:
        data["input_output"] = taco_data["input_output"]
    data["difficulty"] = taco_data["difficulty"]
    data["tags"] = taco_data["tags"]
    data["skill_types"] = taco_data["skill_types"]
    data["raw_tags"] = taco_data["raw_tags"]
    data["date"] = taco_data["date"]
    data["Expected Auxiliary Space"] = taco_data["Expected Auxiliary Space"]
    data["Expected Time Complexity"] = taco_data["Expected Time Complexity"]

    with open(filename, 'w') as file:
        json.dump(data, file)


In [ ]:
def get_taco_data(taco_id):
    if taco_id.startswith("train_"):
        real_id = int(taco_id[6:])
        return taco_dataset["train"][real_id]
    elif taco_id.startswith("test_"):
        real_id = int(taco_id[5:])
        return taco_dataset["test"][real_id]
    else:
        raise ValueError()


In [ ]:
# Copy files to new dataset
import shutil
import os
import json

destination_folder="/home/kaixin/Desktop/mmcode/mmcode_dataset"
for item in deduped_filtered_data:
    taco_id = item["taco_id"]
    crawled_path = item["crawled_path"]

    if "codeforces" in crawled_path or "aizu" in crawled_path:
        continue

    # Use the last folder name of the crawled_path as the new folder name
    new_folder_name = os.path.basename(os.path.normpath(crawled_path))
    new_folder_path = os.path.join(destination_folder, new_folder_name)

    # Create the new folder
    os.makedirs(new_folder_path, exist_ok=True)

    # Copy files from crawled_path to the new folder
    shutil.copytree(crawled_path, new_folder_path, dirs_exist_ok=True)

    # write taco data to a file
    taco_data = get_taco_data(taco_id)
    taco_json_path = os.path.join(new_folder_path, "taco.json")
    with open(taco_json_path, 'w') as f:
        json.dump(taco_data, f, indent=4)

    